# Train the LoRA extractor on a free GPU

Works on Google Colab (Runtime, Change runtime type, T4 GPU) and on Kaggle (Settings, Accelerator, GPU T4).
It trains a LoRA adapter on the training split, scores it on the validation and test splits next to the rules baseline (the LLM row stays pending unless you set ANTHROPIC_API_KEY),
and zips the adapter and the result files for download. A full run on a T4 takes about 10 to 20 minutes for the 0.5B model.

Steps after the run: download `lora_run.zip`, unzip it inside your clone of the repo, run
`python -m filing_extractor.readme_results`, then commit `results/` and the README.

In [ ]:
import os, subprocess
REPO = "https://github.com/Srijareddy123/filing-extractor"  # this repo's URL
if not os.path.exists("filing-extractor"):
    subprocess.run(["git", "clone", "--depth", "1", REPO], check=True)
os.chdir("filing-extractor")

In [ ]:
!nvidia-smi
# Colab already ships torch, transformers and accelerate: do not reinstall them.
!pip install -q -r requirements.txt peft==0.21.2
# Remove torchvision and torchao before training.
!pip uninstall -y -q torchvision torchao
!pip install -q -e . --no-deps

In [ ]:
# Pick the model. 0.5B is fast; 1.5B usually scores higher and still fits a T4.
MODEL = "Qwen/Qwen2.5-0.5B-Instruct"   # or "Qwen/Qwen2.5-1.5B-Instruct"
OUT = "outputs/lora-" + MODEL.split("/")[-1].lower()
!python -m filing_extractor.train_lora --model $MODEL --out $OUT --epochs 3

In [ ]:
# Score the adapter and the rules baseline on val and test. Writes results/metrics.json and results/results.md.
!python -m filing_extractor.evaluate --models rules llm lora --splits val test --lora-adapter $OUT
!python -m filing_extractor.errors --model lora --splits val test
!cat results/results.md

In [ ]:
import shutil
shutil.make_archive("lora_run", "zip", ".", "results")
shutil.make_archive("lora_adapter", "zip", OUT)
try:
    from google.colab import files
    files.download("lora_run.zip"); files.download("lora_adapter.zip")
except Exception:
    print("Download lora_run.zip and lora_adapter.zip from the file browser (Kaggle: Output tab).")